# Notebook 1 - What Is Machine Learning?

## Learning objectives

By the end of this notebook you should be able to:

- State what a dataset, a feature, and a label are.
- Explain in one sentence what it means for a model to *learn*.
- Distinguish supervised, unsupervised, and reinforcement learning, and within supervised learning, regression from classification.
- Walk through the seven stages of the ML pipeline on a given air-quality dataset.
- Apply standardisation and one-hot encoding correctly (i.e., fit on training data only) using `sklearn.Pipeline`.
- Make a chronological train/test split and explain why it is the only correct way to split data for autocorrelated data.

## 1.1 The fundamental object: a dataset

Before we can talk about *learning*, we must define the object that is learned from. A **dataset** is a finite collection of observations. Each observation is a tuple of recorded values describing a single moment, place, or entity.

In the **Beijing Multi-Site Air-Quality Dataset** that we use throughout this notebook series, one observation corresponds to one **station-hour** - a single station (e.g., *Aotizhongxin*) at a single date-time stamp (e.g., 2013-03-01 00:00). For that station-hour the dataset records 17 fields, including:

- year, month, day, hour, station name;
- six pollutant concentrations: $\text{PM}_{2.5}$, $\text{PM}_{10}$, $\text{SO}_{2}$, $\text{NO}_{2}$, $\text{CO}$, $\text{O}_{3}$;
- five meteorological variables: temperature (`TEMP`, °C), pressure (`PRES`, hPa), dew-point temperature (`DEWP`, °C), rainfall (`RAIN`, mm), wind speed (`WSPM`, m/s);
- one categorical variable: wind direction (`wd`, e.g. NNW, ESE).

Across 12 stations and four years, the raw archive contains roughly $12 \times 35\,064 \approx 4.2 \times 10^{5}$ rows.

We define a generic dataset of size $N$ as

$$\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^{N},$$

where $\mathbf{x}_i \in \mathbb{R}^{d}$ is a $d$-dimensional **feature vector** describing observation $i$, and $y_i$ is the **label** or **target** we wish to predict.

> **Definition (Feature).** A feature, also called a *predictor* or *covariate*, is a measurable property of an observation used as input to a model.
>
> **Definition (Label / Target).** The label is the quantity the model is trained to predict.

Let's load the data and see this concretely.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('data')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
print('Data folder:', DATA_DIR.resolve())

## Data files used in this notebook series

All eight notebooks read from the sibling `data/` folder. The five files are:

| File | Format | Used by |
|------|--------|---------|
| `air_quality_basic_aotizhongxin.csv` | wide CSV, 34,139 hourly rows, 27 columns | Notebooks 1, 2, 3 |
| `air_quality_basic_nongzhanguan.csv` | wide CSV, hourly, 27 columns | Notebook 1 exercise |
| `air_quality_nongzhanguan_forecast.csv` | hourly, ~33,000 rows, 50+ feature columns + 3 horizon targets for $PM_{2.5}$ | Notebooks 4–8 |
| `air_quality_nongzhanguan_o3_forecast.csv` | same shape, target is $O_3$ | Notebooks 5, 7, 8 |
| `PRSA_Data_Aotizhongxin_20130301-20170228.csv` | raw hourly, 18 columns, contains literal `NA` for missing values | Notebook 1 missing-data demo |

**`basic_*.csv`** files: one row per hour and per *target station*; the columns are $PM_{2.5}$ and $PM_{10}$ from the eleven *other* stations, plus three target columns (`target_pm25`, `target_category_3`, `target_category_6`). The pre-computed categorical targets are simple $PM_{2.5}$-threshold bins.

**`*_forecast.csv`** files: one row per hour at a single station; columns are current-hour pollutants and meteorology, cyclic-encoded time-of-day / month / day-of-week, and lag features (`PM2.5_lag_1`, `PM2.5_lag_24`, …). Three pre-computed shifted targets give the value of the pollutant 1 hour, 6 hours and 12 hours into the future — labelled e.g. `target_pm25_h1`.

**`PRSA_Data_*.csv`**: the original, raw archive for one station, four years (2013–2017). Used in this notebook to inspect missing-data patterns before any cleaning.

If the data files are missing, ask your instructor for them; the file `setup_data.py` next to the notebooks shows how they were generated.

### Look at one row

The basic teaching dataset for *Aotizhongxin* contains 22 candidate features ($\text{PM}_{2.5}$ and $\text{PM}_{10}$ from 11 other stations), three targets, and a datetime index.

In [ ]:
df = pd.read_csv(DATA_DIR / 'air_quality_basic_aotizhongxin.csv', parse_dates=['datetime'])
print('Shape:', df.shape)
print('Date range:', df['datetime'].min(), '->', df['datetime'].max())
df.head()

## 1.2 What does it mean to "learn"?

A machine learning algorithm learns by searching, within a chosen family of mathematical functions $\mathcal{F}$, for the function $\hat{f} \in \mathcal{F}$ that best predicts $y$ from $\mathbf{x}$ on data it has seen, while still generalising to data it has not seen. Formally, given a **loss function** $L(\hat{y}, y)$ measuring how wrong a prediction $\hat{y}$ is compared with the truth $y$, the algorithm seeks

$$\hat{f} = \arg\min_{f \in \mathcal{F}} \; \frac{1}{N} \sum_{i=1}^{N} L\bigl(f(\mathbf{x}_i),\, y_i\bigr).$$

The art of machine learning lies in three choices: (i) which **family** $\mathcal{F}$ to search within (linear functions, decision trees, neural networks…), (ii) which **loss** $L$ to optimise (squared error, cross-entropy, …), and (iii) how to ensure the chosen $\hat{f}$ generalises beyond the training set.

## 1.3 Types of learning

Machine learning is conventionally divided into three regimes:

- **Supervised learning.** Each training observation has a known label $y_i$. The model learns a mapping $\mathbf{x} \to y$. *All examples in this notebook series are supervised.* Predicting $\text{PM}_{2.5}$ at *Aotizhongxin* from the $\text{PM}_{2.5}$ values at 11 other stations is supervised learning, because for every training row we know the true $\text{PM}_{2.5}$ at *Aotizhongxin*.
- **Unsupervised learning.** The training data has no labels. The model finds structure - clusters, low-dimensional embeddings, density estimates. Example: grouping the 12 Beijing stations into "urban", "suburban" and "town" clusters using only their pollutant time series.
- **Reinforcement learning.** An agent interacts with an environment, receiving rewards for actions, and learns a policy that maximises long-term reward.

Within **supervised learning**, we distinguish two further subtypes by the nature of $y$:

- **Regression**: $y$ is continuous (e.g., $\text{PM}_{2.5}$ in $\mu\text{g}/\text{m}^{3}$).
- **Classification**: $y$ is categorical (e.g., $\text{PM}_{2.5}$ category $\in \{\text{Low}, \text{Medium}, \text{High}\}$).

The same input data can be framed as either problem. The cell below shows the *same* hourly $\text{PM}_{2.5}$ measurement supporting all three framings used throughout this course.

In [ ]:
# Three targets built from the same $\text{PM}_{2.5}$ measurement
df[['datetime', 'target_pm25', 'target_category_3', 'target_category_6']].head(10)

Notice that `target_pm25` is continuous (regression target) while `target_category_3` and `target_category_6` are categorical (classification targets) - the **same row** can be modelled either way. The categorical labels are produced from the continuous $\text{PM}_{2.5}$ by simple binning, e.g:

```python
def pm25_category_3(pm25):
    bins = [-float('inf'), 35, 75, float('inf')]
    return pd.cut(pm25, bins=bins, labels=['Low', 'Medium', 'High'])

def pm25_category_6(pm25):
    bins = [-float('inf'), 35, 75, 115, 150, 250, float('inf')]
    return pd.cut(pm25, bins=bins, labels=['Excellent', 'Good', 'Lightly Polluted', 'Moderately Polluted', 'Heavily Polluted', 'Severely Polluted'])
```

**Key point**: whether a problem is regression or classification is a *modelling choice*, not a property of the data.

## 1.4 Examples of ML problems in atmospheric science

Five concrete examples drawn from this dataset. None require Python to understand; we only need to know what is measured, what is predicted, and which type of learning applies.

1. **Hourly $\text{PM}_{2.5}$ nowcasting.**  
Inputs: $\text{PM}_{2.5}$ and $\text{PM}_{10}$ measured *right now* at 11 Beijing stations.  
Output: $\text{PM}_{2.5}$ at a 12th, target station, *right now*.  
Task: regression.
2. **Air-quality category labelling.**  
Inputs: same as above.  
Output: the qualitative category of $\text{PM}_{2.5}$ at the target station - *Low / Medium / High*.  
Task: 3-class classification.
3. **One-hour-ahead $\text{NO}_{2}$ forecast at *Nongzhanguan*.** Inputs: the last 24 hours of pollutants and meteorology at *Nongzhanguan*.  
Output: $\text{NO}_{2}$ one hour into the future.  
Task: regression with a temporal target.
4. **Twelve-hour-ahead $\text{O}_{3}$ forecast.** 
Same inputs as (3); output is $\text{O}_{3}$ at $t+12$.  
Task: regression, harder horizon.
5. **Pollution event detection.**  
Output: binary indicator of whether $\text{PM}_{2.5}$ will exceed $150~\mu\text{g}/\text{m}^{3}$ in the next 6 hours.  
Task: binary classification.

Examples 1, 3 and 4 are regression. Examples 2 and 5 are classification. All five are supervised.

## 1.5 The ML learning pipeline

Real ML projects follow a recurring sequence of stages. Each stage feeds the next; an error early in the pipeline silently degrades everything downstream.

```
data collection → data cleaning → feature engineering → data splitting → model training → model validation → model deployment
```

We shall analyse each stage in turn.

### Class distribution - a first look at the labels

Atmospheric categories are **highly imbalanced**. *Low* hours dominate (clean nights and early mornings), *Severely Polluted* hours are rare. A naïve classifier that always predicts *Low* would still achieve a high accuracy purely from base-rate dominance - exposing why we will need class-aware metrics in Notebook 3.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['target_category_3'].value_counts().reindex(['Low', 'Medium', 'High']).plot(
    kind='bar', ax=axes[0], color='steelblue', title='3-class distribution')
axes[0].set_ylabel('rows')
labels6 = ['Excellent', 'Good', 'Lightly Polluted', 'Moderately Polluted', 'Heavily Polluted', 'Severely Polluted']
df['target_category_6'].value_counts().reindex(labels6).plot(
    kind='bar', ax=axes[1], color='darkorange', title='6-class distribution')
axes[1].set_ylabel('rows')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### Stage 1: Data collection

The starting point. Sensor networks, satellite retrievals, model reanalyses, manually digitised archives. The Beijing Multi-Site dataset originates from 12 fixed monitoring stations operated under the Chinese Ministry of Ecology and Environment and was published as a benchmark in 2017. Hourly observations span four full annual cycles, which is enough to capture seasonality.

Quality at this stage matters. Sensor calibration errors, transmission gaps, station-relocation events, and unit changes (e.g., switching from $\mu\text{g}/\text{m}^{3}$ at 0 °C 1013 hPa to ambient conditions) all translate into model bias if uncaught.

### Stage 2: Cleaning and missing-data handling

Real atmospheric data is never complete. Sensors fail; transmissions drop; routine maintenance creates outages.Each station in the multi-site dataset has its own missing-data pattern. In some cases, a sensor malfunction at one station may last several days while other stations continue recording normally. This creates an important opportunity: when the target station is missing, data from nearby stations may still be used to impute it.  

**Deletion**: Remove rows where the target $\text{PM}_{2.5}$ is missing. Safe only if missingness is random and less than 5%. For urban-core stations, missing $\text{PM}_{2.5}$ may be more frequent during severe haze events due to sensor saturation, so naive deletion biases the dataset toward cleaner conditions.

**Forward Fill (Last Observation Carried Forward - LOCF)**: Replace a missing $\text{PM}_{2.5}$ with the most recent valid value at the same station. Reasonable for gaps under 3 hours, but creates artificially flat segments during longer outages. Note that this method is suitable for timeseries data, definitely not for data such as a Housing Price Dataset which consists of various indices and metrics related to housing prices.  

**Spatial imputation**: Unique to multi-site data. Replace a missing $\text{PM}_{2.5}$ at station $A$ at time $t$ with a weighted average of PM2.5 from nearby stations at the same time. Weights reflect physical distance or historical correlation. This is more accurate than LOCF during abrupt pollution events when the temporal signal is less reliable than the spatial signal.
  
**Model-based imputation**: Train a regression model using the other 11 stations' $\text{PM}_{2.5}$ values as features to predict the missing station's value. Sophisticated but computationally intensive.

The Beijing dataset uses the literal token `NA` for missing observations. The standard read pattern handles this explicitly:

```python
df = pd.read_csv(path, na_values=['NA'])
```

After detection, we must decide what to do with each missing value. Three strategies are common:

- **drop** - discard any row that contains a missing value.
- **interpolate** - linearly interpolate across short gaps only (e.g., max 3 hours).
- **ffill_short** - forward-fill across short gaps only.

The reasoning behind the gap limit is **physical**: $\text{PM}_{2.5}$ varies smoothly across one or two hours, so filling a 1- or 2-hour outage by interpolation is reasonable. A 24-hour outage, by contrast, can span an entire pollution episode; filling it would manufacture data that looks like a smooth ramp where reality contained a sharp pollution event.

In [ ]:
raw = pd.read_csv(
    DATA_DIR / 'PRSA_Data_Aotizhongxin_20130301-20170228.csv',
    na_values=['NA'])
print('Raw rows:', len(raw))
print('\nMissing values per column (top 10):')
print(raw.isna().sum().sort_values(ascending=False).head(10))

### Stage 3: Feature engineering

Raw measurements rarely make the best inputs. **Feature engineering** is the deliberate construction of new columns that expose structure already present in the data, but in a form a model can exploit. We perform five distinct feature-engineering operations, each covered in detail in **Notebook 4**:

1. **Cyclic encoding** of the hour-of-day, month-of-year, and day-of-week.
2. **Lag features**: prior-hour, prior-6-hour, prior-12-hour and prior-24-hour values of each pollutant.
3. **One-hot encoding** of the categorical wind direction `wd`.
4. **Standardisation** of all numeric features to zero mean and unit variance.
5. **Cross-station alignment**: turning 12 long-format station files into one wide-format table indexed by `datetime`.

### Stage 4: Splitting into training, validation and test sets

A model that has seen all the data cannot be trusted: it may have *memorised* rather than *learned*. We therefore set aside a portion of the dataset, the **test set**, that the model never sees during fitting.

For time-ordered data such as ours, the only honest split is **chronological**: train on early data, test on later data. A random split would let information from the future leak into training. We will demonstrate the leakage in Notebook 2.

A third “validation” set is often carved out from the training data and used for model-selection decisions (e.g., choosing a hyper-parameter). When using cross-validation (CV) (Section 6.4), the validation set is created automatically by the CV procedure.
**Training set**: The data on which the model parameters are optimized. Typically, 60-80% of total data.
**Validation set**: Used to tune hyperparameters and select between models. Typically, 10-20%.
**Test set**: Used exactly once at the very end to estimate real-world performance. Never used during training or model selection. Typically, 10-20%.




For time-ordered data the only honest split is chronological - the last 20% becomes the test set, never randomly drawn:

In [ ]:
df_sorted = df.sort_values('datetime').reset_index(drop=True)
cut = int(0.8 * len(df_sorted))
train_df, test_df = df_sorted.iloc[:cut], df_sorted.iloc[cut:]
print(f'Train: {len(train_df):,} rows  ({train_df.datetime.min()} -> {train_df.datetime.max()})')
print(f'Test : {len(test_df):,} rows  ({test_df.datetime.min()} -> {test_df.datetime.max()})')

### Stage 5: Training, validation, deployment

Training executes the optimisation in equation (§1.2). For linear regression there is a closed-form solution (Notebook 2); for boosted trees the solution is iterative. In `scikit-learn` this stage reduces to  
`model.fit(x_train, y_train)`.
Conceptually, training is the act of modifying the model’s internal parameters until the loss on the training set is minimised, subject to constraints that prevent over-fitting.

### Stage 6: Validation and evaluation
After training, we compute predictions on the held-out test set and summarise their quality with one or more **metrics**: MAE, RMSE, $R^{2}$ for regression (Notebook 2); accuracy, $F_{1}$, confusion matrix for classification (Notebook 3).

### Stage 7: Deployment
The final stage. A trained model is saved (e.g., serialised to disk) and integrated into a production pipeline. Deployment also requires *monitoring*: the data distribution may drift over time (a new factory opens; a road is closed) and the model must be retrained periodically.

## 1.6 Feature scaling

Many ML algorithms - linear regression with gradient-based solvers, k-nearest neighbours, support vector machines, neural networks - are sensitive to the *scale* of the inputs. If `PRES` ranges from 980 to 1030 hPa and `RAIN` ranges from 0 to 0.5 mm in the same dataset, then naive distance-based methods will treat `PRES` as ~100 times more important than `RAIN` purely because of its numeric range.

The standard remedy is **standardisation**:

$$\tilde{x}_{ij} = \frac{x_{ij} - \mu_{j}}{\sigma_{j}},$$

where $\mu_{j}$ and $\sigma_{j}$ are the mean and standard deviation of feature $j$, computed *on the training set*. After standardisation every feature has mean $0$ and standard deviation $1$.

> **Critical rule:** the scaler is fitted **on training data only**. If we computed $\mu$ and $\sigma$ on the entire dataset, statistics from the test set would leak into training.

## 1.7 Encoding categorical features

Categorical encoding is the essential transformation of qualitative data (such as categories, labels, or strings) into numerical formats that machine learning algorithms can process mathematically. Selecting the appropriate approach depends on the nature of your categories and the requirements of your chosen model.  

For example: Wind direction `wd` takes 16 values: `N`, `NNE`, `NE`, …, `WNW`, `NW`, `NNW`. Numeric algorithms cannot consume strings. The default encoding we use is **one-hot** where each categorical level becomes a 0/1 indicator column.

```python
OneHotEncoder(handle_unknown='ignore', sparse_output=False)
```

The `handle_unknown='ignore'` argument is essential for time-series deployment: if an unseen wind-direction string appears in production (a sensor glitch, a relabelling), the encoder emits a row of zeros rather than throwing an exception.  

However, wind direction has an inherent cyclic topology: N (north) is adjacent to NNE and NNW, and encoding N=0 and NNW=15 implies they are 15 units apart when they are actually just 22.5 degrees apart. Section 4.5 covers cyclic sine-cosine encoding, which is the correct approach for any directional variable.  
The station column (variable) identifies which of the 12 monitoring sites a row belongs to. Several encoding strategies are available, each with different trade-offs. For example, **One-Hot Encoding** of the station variable creates 12 binary columns. In this approach, each station gets its own indicator, allowing the model to learn station-specific intercepts. Another approach for this variable would be to implement **Target Encoding (Mean Encoding)**. In this approach, the station name is replaced with the mean PM2.5 of that station computed from the training set. This is compact (we have only 1 column) and informative, but requires strict application to training data only to prevent target leakage.


### The pipeline in three lines

A scaler + a model wrapped in a `Pipeline` enforces *fit on training data only* automatically - the single most important leak-prevention pattern.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])
print(pipe)

When `pipe.fit(x_train, y_train)` runs, `StandardScaler` is fitted on `x_train` only, and the resulting $\mu, \sigma$ are reused at prediction time on `x_test`. There is no way to accidentally peek at the test data.

## 1.8 Common misconceptions

**"More data is always better."** Not in time series. If a sensor was relocated in 2015 and its measurements jumped by 20 %, then including the 2013–2014 data without an adjustment makes things *worse*, not better. Domain understanding always precedes data quantity.  

**"ML can predict anything if we have enough features."** ML can only predict what is *informative* in the data. $\text{SO}_{2}$ at *Nongzhanguan* twelve hours from now is only weakly predictable from current weather alone; $\text{SO}_{2}$ is dominated by industrial point-source episodes that no atmospheric feature exposes.  

**"The pipeline runs once."** In production it runs continuously. Concept drift, sensor changes, seasonality shifts, all force periodic retraining.  

**"Splitting randomly is fine because each row is independent."** Atmospheric rows are *not* independent: $\text{PM}_{2.5}$ at hour $t$ is strongly correlated with $\text{PM}_{2.5}$ at hour $t-1$. A random split with autocorrelated data leaks future information into training. Notebook 4 returns to this.

## 1.9 Chapter summary

- A dataset is a finite sample of $(\mathbf{x}_i, y_i)$ pairs; ML is the search, within a chosen family of functions, for the one that minimises a chosen loss while still generalising to unseen data.
- Supervised learning subdivides into **regression** (continuous $y$) and **classification** (categorical $y$); the same Beijing $\text{PM}_{2.5}$ measurement supports both framings.
- The ML pipeline is data collection → data cleaning → feature engineering → data splitting → model training → model validation → deployment; missing-data, scaling and encoding decisions are handled in the feature-engineering stage.
- Standardisation, one-hot encoding, and the train-only fitting rule are non-negotiable defaults that we enforce through `sklearn.Pipeline`.
- Time-ordered atmospheric data demands a **chronological** train/test split; random splits leak future information and overstate model quality.

**Next:** Notebook 2 fits a real linear regression on this data and computes the headline regression metrics.